#### [MNiST 손글씨 숫자 이미지 분류] <hr>

- 데이터 : mnist_train.csv, mnisi_test.csv
- 내 용 : 손글씨 그림을 저장한 파일. 0 ~ 9까지 숫자 데이터
- 주 제 : 숫자 0 ~ 9 이미지를 전달해서 정확하게 분류해주는 모델
- 학습종류 : 지도학습 + 분류
- 학습방법 : KNN + SVC

**[0] 모듈 로딩**

In [ ]:
## 데이터 관련
import pandas as pd
import numpy as np

## 시각화 관련
import matplotlib.pyplot as plt, koreanize_matplotlib
import seaborn as sns

## ML 교차검증 관련
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

## ML 데이터 전처리 관련
from sklearn.preprocessing import MinMaxScaler

## ML 모델 관련
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.pipeline import Pipeline

import joblib, os

**[1] 데이터 준비 및 확인**

In [ ]:
train = pd.read_csv('../Data/Numbers/mnist_train.csv', header=None)
test = pd.read_csv('../Data/Numbers/mnist_test.csv', header=None)
MODE_FILE = '../Models/mnist_model.pkl'

In [ ]:
train.info()
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Columns: 785 entries, 0 to 784
dtypes: int64(785)
memory usage: 359.3 MB
<class 'pandas.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Columns: 785 entries, 7 to 0.667
dtypes: int64(785)
memory usage: 59.9 MB


**[2] 학습용 | 테스트용 데이터 준비**

In [10]:
# 피쳐 타겟 분리
x_train = train[train.columns[1:]]
y_train = train[train.columns[0]]

x_test = test[test.columns[1:]]
y_test = test[test.columns[0]]

print(f'[Train] {x_train.shape}, {y_train.shape}')
print(f'[Test] {x_test.shape}, {y_test.shape}')

[Train] (60000, 784), (60000,)
[Test] (9999, 784), (9999,)


**[3] 데이터 분석 및 전처리**
- 클래스 균형/불균형 체크
- 피쳐 전처리 방법 결정

In [ ]:
# => 클래스 균형-불균형 체크
print(f'[Train Class]\n{y_train.value_counts().to_list()}') 
print(f'{y_train.value_counts(normalize=True).round(2).to_list()}')

# => 클래스별 비율 → 균형 데이터 (별도 처리 불필요)
print(f'\n[Test Class]\n{y_test.value_counts().to_list()}')
print(f'{y_test.value_counts(normalize=True).round(2).to_list()}')

In [20]:
# => 1개 행만 추출 피쳐 확인
x_train.iloc[0].describe()

# => 0 ~ 255 범위 값 ==> 0 ~ 1 사이로 스케일링 : MinMaxScaler

count    784.000000
mean      35.108418
std       79.699674
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max      255.000000
Name: 0, dtype: float64

**[4] 전처리 + 교차검증 + 튜닝**

In [ ]:
# 전처리 인스턴스
scaler = MinMaxScaler()

# 모델 인스턴스
knn = KNeighborsClassifier()

# 파이프라인 인스턴스 (전처리 → 모델 자동 연결)
pipe = Pipeline(steps=[
    ('scaler', scaler),
    ('model',  knn)
])

# 모델 파라미터 Dict (파이프라인 파라미터명 형식: 단계이름__파라미터명)
param_dict = {
    'model__n_neighbors': range(1, 10, 2),    # 1, 3, 5, 7, 9
    'model__weights':     ['uniform', 'distance'],
    'model__p':           [1, 2]              # 1=맨해튼거리, 2=유클리드거리
}

# 교차검증 인스턴스
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 튜닝 인스턴스 (GridSearchCV)
gs = GridSearchCV(pipe, param_grid=param_dict,
                  cv=skf, return_train_score=True, n_jobs=-1)
gs.fit(x_train, y_train)

# 최적화된 인스턴스
best_model = gs.best_estimator_
print(f'최적 파라미터   : {gs.best_params_}')
print(f'최고 교차검증 점수 : {gs.best_score_:.4f}')

**[5] 성능 평가**

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 테스트 데이터 예측
y_pred = best_model.predict(x_test)

# Train / Test 점수
print(f'Train Score : {best_model.score(x_train, y_train):.4f}')
print(f'Test  Score : {best_model.score(x_test,  y_test):.4f}')
print(f'Accuracy    : {accuracy_score(y_test, y_pred):.4f}')
print()

# 클래스별 분류 리포트
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix 시각화
cm   = confusion_matrix(y_test, y_pred)
cmDF = pd.DataFrame(cm, index=range(10), columns=range(10))

plt.figure(figsize=(10, 8))
sns.heatmap(cmDF, annot=True, fmt='d', cmap='Blues')
plt.xlabel('예측 숫자')
plt.ylabel('실제 숫자')
plt.title('MNIST Confusion Matrix')
plt.show()

In [ ]:
# 모델 저장
os.makedirs('../Models', exist_ok=True)
joblib.dump(best_model, MODE_FILE)
print(f'모델 저장 완료 : {MODE_FILE}')

# 저장된 모델 불러오기
loaded_model = joblib.load(MODE_FILE)
print(f'불러온 모델 테스트 점수 : {loaded_model.score(x_test, y_test):.4f}')